# MiniGPT from scratch — Colab runner
This notebook clones the repository and runs everything end to end: tests → architecture evidence → tiny-overfit gate → training → samples → attention maps → ablations → API check.

**Before you start:** `Runtime → Change runtime type → GPU (T4)`. Then run the cells top to bottom.
Set `REPO_URL` below to your GitHub repository.

In [ ]:
REPO_URL = "https://github.com/<your-username>/Small-Language-Model.git"   # <-- CHANGE THIS
QUICK = False          # True = short smoke run (a few minutes); False = full training
RUN_ABLATIONS = True   # the 3 required experiments (depth / heads / context) -- takes the longest
ABLATION_STEPS = 3000  # steps per ablation run
MAIN_STEPS = 5000      # steps for the main model

In [ ]:
import os, subprocess, torch
if not os.path.exists("Small-Language-Model"):
    !git clone $REPO_URL Small-Language-Model
%cd Small-Language-Model
!pip install -q -r requirements-train.txt
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Unit tests (attention, masks, shapes, overfit, checkpoint reload, API)

In [ ]:
!python -m pytest -q

## 2. Architecture evidence (Figures 1–5)

In [ ]:
!python experiments/hand_calculation.py
!python experiments/shape_trace.py
!python experiments/positional_encoding_viz.py
!python experiments/architecture_diagram.py
from IPython.display import Image, display
for f in ["fig1_hand_attention", "fig2_causal_mask", "fig4_positional_encoding", "architecture_diagram"]:
    display(Image(f"outputs/figures/{f}.png"))

## 3. Why Transformers (sequential vs parallel) and MLM-vs-CLM comparison

In [ ]:
!python experiments/why_transformers.py
!python experiments/mask_comparison.py
display(Image("outputs/figures/why_transformers.png")); display(Image("outputs/figures/fig6c_mlm_vs_clm_training.png"))

## 4. Debugging gate: overfit a tiny repeated batch (loss must collapse)

In [ ]:
!python -m src.train --overfit_tiny

## 5. Train the main MiniGPT

In [ ]:
steps = 300 if QUICK else MAIN_STEPS
eval_i = 250 if steps >= 1000 else 50
samp_i = 1000 if steps >= 1000 else 100
!python -m src.train --set train.max_steps={steps} train.eval_interval={eval_i} train.sample_interval={samp_i}

In [ ]:
import json
display(Image("outputs/figures/main_loss_curve.png"))
print(json.dumps(json.load(open("outputs/logs/main_summary.json")), indent=2)[:1800])

## 6. Generation (fixed prompts), evaluation, attention maps

In [ ]:
!python -m src.generate --fixed-prompts --max-new-tokens 200
!python experiments/evaluate.py --checkpoint checkpoints/best.pt --device cuda
!python experiments/attention_visualization.py --checkpoint checkpoints/best.pt --device cuda
display(Image("outputs/figures/fig7_attention_visualization.png"))

## 7. Ablation study (depth / heads / context length) + generation strategy

In [ ]:
if RUN_ABLATIONS:
    s = 200 if QUICK else ABLATION_STEPS
    !python experiments/ablation.py --group depth heads context --steps $s
    !python experiments/ablation.py --group generation --checkpoint checkpoints/best.pt
    for g in ["depth", "heads", "context"]:
        display(Image(f"outputs/figures/fig9_ablation_{g}.png"))
    print(open("outputs/ablation/ablation_results.md").read())

## 8. API smoke test (same app that is deployed on Render)

In [ ]:
import subprocess, time, requests
server = subprocess.Popen(["uvicorn", "src.api:app", "--port", "8000"])
time.sleep(8)
print(requests.get("http://127.0.0.1:8000/health").json())
r = requests.post("http://127.0.0.1:8000/generate", json={"prompt": "ROMEO:", "max_new_tokens": 120, "temperature": 0.8})
print(r.json()["generated"])
server.terminate()

## 9. Save results (download the zip, or copy to Drive) and push `checkpoints/best.pt` + `outputs/` to GitHub

In [ ]:
!zip -qr results.zip outputs checkpoints/best.pt
from google.colab import files
files.download("results.zip")
# Optional: copy to Google Drive
# from google.colab import drive; drive.mount('/content/drive'); !cp -r outputs checkpoints/best.pt /content/drive/MyDrive/minigpt_results/